# Homework #2 - Sistemi Informativi Evoluti e Big Data (2025-2026)
## Analisi dei dati criminali di Los Angeles - InfluxDB

Questo notebook implementa l'analisi completa del dataset [Crime Data from 2020 to Present](https://catalog.data.gov/dataset/crime-data-from-2020-to-present) utilizzando InfluxDB.

## Setup - Installazione delle librerie necessarie

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
from datetime import datetime

# Librerie InfluxDB
from influxdb_client import InfluxDBClient, Point, WriteOptions
from influxdb_client.client.write_api import SYNCHRONOUS

# Librerie per Clustering 
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')


## 1. Configurazione e Caricamento Dati

Configuriamo la connessione a InfluxDB e carichiamo il dataset CSV.

In [ ]:

# Configurazione InfluxDB
url = "http://localhost:8086"
token = "W3cPRGkq7ds7JHFNtuciiVqEVPGugFye82mykiLWWee2FMxLDflsqsHk8OJaavTS9Sjh1_H31xU4yh3J5IC9RA=="
org = "unibs"
bucket = "crime_data"

client = InfluxDBClient(url=url, token=token, org=org, timeout=3000000)
write_api = client.write_api(
    write_options=WriteOptions(
        batch_size=5_000,        # numero di punti per batch
        flush_interval=5_000,  # flush automatico ogni 10 secondi
        jitter_interval=2_000,  # aggiunge un po' di random per evitare spike
        retry_interval=5_000,   # ritenta in caso di errore temporaneo
    )
    )
query_api = client.query_api()

In [ ]:
from influxdb_client.rest import ApiException

buckets_api = client.buckets_api()
# 1. Cerca il bucket esistente
try:
    existing_bucket = buckets_api.find_bucket_by_name(bucket)
    
    if existing_bucket:
        print(f"Trovato bucket esistente (ID: {existing_bucket.id}). Eliminazione in corso...")
        # 2. Cancella il bucket
        buckets_api.delete_bucket(existing_bucket)
        print("Bucket eliminato.")
    else:
        print("Il bucket non esisteva.")

    # 3. Aspetta un attimo per sicurezza (InfluxDB è veloce ma non istantaneo)
    time.sleep(2)

    # 4. Crea il bucket nuovo di zecca
    # Cerchiamo l'ID della tua organizzazione per creare il bucket al posto giusto
    org_api = client.organizations_api()
    org_obj = org_api.find_organizations(org=org)[0]
    
    print(f"Creazione nuovo bucket '{bucket}' nell'organizzazione '{org}'...")
    created_bucket = buckets_api.create_bucket(bucket_name=bucket, org_id=org_obj.id)
    print(f"✅ Bucket creato con successo (Nuovo ID: {created_bucket.id})")

except ApiException as e:
    print(f"❌ Errore durante il reset: {e}")
except Exception as e:
    print(f"❌ Errore generico: {e}")

# 5. Verifica Finale (Deve essere 0)
print("\nVerifica stato vuoto...")
q_check = f'from(bucket: "{bucket}") |> range(start: 1970-01-01T00:00:00Z) |> count()'
try:
    result = query_api.query(q_check)
    if not result:
        print(f"✅ CONFERMATO: Il bucket '{bucket}' è perfettamente VUOTO (0 record).")
        print("Puoi procedere con il caricamento.")
    else:
        print("⚠️ QUALCOSA NON VA: Il bucket contiene ancora dati!")
except Exception as e:
    # Se il bucket è vuoto, a volte la query count() non ritorna nulla o da errore su range vuoti
    # Se non trova tabelle, è vuoto.
    print(f"✅ CONFERMATO: Il bucket è vuoto (Nessuna tabella trovata).")

## 1. Caricamento Dati e Preprocessing

Carichiamo il Dataset e lo prepariamo al caricamento.

In [ ]:
# Caricamento Dataset
csv_path = "Crime_Data_from_2020_to_Present.csv"
df = pd.read_csv(csv_path)

# Conversione date (necessaria per poter fare i confronti nel loop)
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
# df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')

Definiamo funzioni di supporto per pulire i dati secondo le specifiche e arricchire il dataset con nuove feature (Stagione, Giorno/Notte).

In [ ]:
def get_season(date):
    month = date.month
    if month in [3, 4, 5]: return "Spring"
    elif month in [6, 7, 8]: return "Summer"
    elif month in [9, 10, 11]: return "Autumn"
    else: return "Winter"

# def get_season_precise(date):
#     # Tupla (mese, giorno)
#     md = (date.month, date.day)
#     if (3, 21) <= md < (6, 21):
#         return "Spring"
#     elif (6, 21) <= md < (9, 21):
#         return "Summer"
#     elif (9, 21) <= md < (12, 21):
#         return "Autumn"
#     else:
#         # 12, 21 - 3, 20
#         return "Winter"

def get_day_night(time_occ):
    hour = int(time_occ / 100) # TIME OCC è un intero (es. 1430 -> 14:30)
    return "Day" if 6 <= hour < 18 else "Night"

# Elabora il df riga per riga simulando un flusso di dati
def data_generator(df_input):
    cnt_coords = 0
    cnt_date = 0
    cnt_crm = 0
    cnt_written = 0

    cutoff_date = pd.Timestamp('2025-11-30')
    
    for index, row in df_input.iterrows():
        # Coordinate Nulle
        if row['LAT'] == 0 and row['LON'] == 0:
            cnt_coords += 1
            continue
            
        # Date Future o Nulle
        if pd.isna(row['DATE OCC']) or row['DATE OCC'] > cutoff_date:
            cnt_date += 1
            continue
            
        # Tipo di Crimine non valido
        if pd.isna(row['Crm Cd']):
            cnt_crm += 1
            continue

        # feature engineering
        season = get_season(row['DATE OCC'])
        day_night = get_day_night(row['TIME OCC'])

        # Pulizia
        if pd.notna(row['Vict Sex']) and row['Vict Sex'] in ['M', 'F']:
            vict_sex = str(row['Vict Sex'])
        else:
            vict_sex = "X"
        
        vict_descent = str(row['Vict Descent']) if pd.notna(row['Vict Descent']) and row['Vict Descent'] != '-' else "X"
        
        # Gestione Arma
        if pd.notna(row['Weapon Used Cd']):
            weapon_flag = "True"
            weapon_detail = str(row['Weapon Desc'])
            
            # Dall'analisi dati è emerso che 'STRONG-ARM' e 'VERBAL THREAT' sono classificati come armi.
            # Se volessimo distinguere le armi vere dalla violenza fisica:
            # if "STRONG-ARM" in weapon_detail.upper() or "VERBAL THREAT" in weapon_detail.upper():
            #     weapon_flag = "False"  # Tecnicamente "Senza Arma Esterna"
            
        else:
            weapon_flag = "False"
            weapon_detail = "None"

        # Creazione punto segnalazione
        point = (
            Point("segnalation")
            .time(row['DATE OCC'])
            .tag("DR_NO", str(int(row['DR_NO']))) # df granularità giornaliera. due crimini == giorno && == distretto -> == Timestamp && == Tag geografico => InfluxDB sovrascrive il primo con il secondo
            .tag("AREA_ID", str(int(row['AREA'])))
            .tag("AREA_NAME", str(row['AREA NAME']))
            .tag("Rpt_Dist_No", str(int(row['Rpt Dist No'])))
            .tag("Part_1_2", str(int(row['Part 1-2'])))
            .tag("Crm_Cd", str(int(row['Crm Cd'])))
            .tag("Crm_Cd_Desc", str(row['Crm Cd Desc']))
            .tag("Vict_Sex", vict_sex)
            .tag("Vict_Descent", vict_descent)
            .tag("WeaponFlag", weapon_flag)
            .tag("WeaponDetail", weapon_detail)
            .tag("Season", season)
            .tag("DayNight", day_night)
            .field("LAT", float(row['LAT']))
            .field("LON", float(row['LON']))
            .field("Date Rptd", str(row['Date Rptd']))  # non si possono avere due tag time
        )
        # Aggiunta campo età vittima solo se valida (>0)
        if pd.notna(row['Vict Age']) and row['Vict Age'] > 0:
            point.field("Vict_Age", float(row['Vict Age']))
        # Scrivi punto su InfluxDB
        write_api.write(bucket=bucket, org=org, record=point)
        cnt_written += 1

    print(f"Segnalazioni scritte: {cnt_written}")
    print(f"Segnalazioni scartate per coordinate nulle: {cnt_coords}")
    print(f"Segnalazioni scartate per data non valida: {cnt_date}")
    print(f"Segnalazioni scartate per codice crimine non valido: {cnt_crm}")

In [ ]:
# Scrittura su InfluxDB
print("Inizio caricamento dati con filtro in tempo reale...")
start_time = time.time()

data_generator(df)
print(f"Tempo totale operazione: {time.time() - start_time:.2f} secondi")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print("--- Interrogazione 1: Conteggio Multidimensionale (Distretto/Classe/Arma) ---")

# FLUX QUERY
# Raggruppiamo per i 4 TAG chiave richiesti:
# 1. Rpt_Dist_No (Distretto)
# 2. Area_Name (Indicazione area appartenenza)
# 3. Part_1_2 (Classificazione: 1=Grave, 2=Meno Grave)
# 4. WeaponFlag (Con/Senza Arma)
q_multi_count = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "segnalation")
  |> filter(fn: (r) => r["_field"] == "LAT")
  |> group(columns: ["AREA_NAME", "Rpt_Dist_No", "Part_1_2", "WeaponFlag"])
  |> count()
  |> rename(columns: {{_value: "Total_Crimes"}})
'''

try:
    print("Esecuzione query di aggregazione (potrebbe richiedere alcuni secondi)...")
    df_counts = query_api.query_data_frame(q_multi_count)
    
    # Se Influx restituisce una lista di DataFrame (comune con group by), li concateniamo
    if isinstance(df_counts, list):
        df_counts = pd.concat(df_counts)
        
    if not df_counts.empty:
        # Selezioniamo e ordiniamo le colonne rilevanti per il report
        cols = ['Area_Name', 'Rpt_Dist_No', 'Part_1_2', 'WeaponFlag', 'Total_Crimes']
        # Convertiamo Total_Crimes in intero
        df_counts['Total_Crimes'] = df_counts['Total_Crimes'].astype(int)
        
        # Ordiniamo per numero di crimini decrescente per vedere i "hotspot"
        final_report = df_counts[cols].sort_values(by='Total_Crimes', ascending=False)
        
        print(f"\nRisultati trovati: {len(final_report)} combinazioni uniche.")
        print("\nTop 15 Gruppi (Distretto/Area - Tipo - Arma):")
        print(final_report.head(15).to_string(index=False))
        
        # --- ANALISI AGGIUNTIVA VISIVA (Opzionale ma consigliata) ---
        # Creiamo una pivot table per visualizzare meglio la distribuzione "Con/Senza Arma" per Area
        # Aggreghiamo per Area per rendere il grafico leggibile (i distretti sono troppi)
        pivot_weapon = final_report.groupby(['Area_Name', 'WeaponFlag'])['Total_Crimes'].sum().unstack()
        
        pivot_weapon.plot(kind='bar', stacked=True, figsize=(15, 6), color=['skyblue', 'salmon'])
        plt.title('Distribuzione Crimini Con/Senza Arma per Area')
        plt.ylabel('Numero Totale Crimini')
        plt.xlabel('Area')
        plt.legend(title='Arma Usata (WeaponFlag)')
        plt.tight_layout()
        plt.show()
        
    else:
        print("Nessun dato trovato con i criteri specificati.")

except Exception as e:
    print(f"Errore durante l'esecuzione della query: {e}")

Alerting

In [ ]:
# Definizione soglie statiche (o calcolate pre-loop se preferisci usare lo storico)
# Per simulazione, usiamo valori fissi o calcolati su tutto il DF per semplicità, 
# ma il controllo avviene riga per riga.
daily_counts = df['DATE OCC'].dt.date.value_counts()
MAX_CRIMES_DAY = daily_counts.max()

# Soglie [cite: 21]
SOGLIA_WARNING = MAX_CRIMES_DAY * 0.05  # Esempio logica: 5% del max (o logica custom)
SOGLIA_ALARM = MAX_CRIMES_DAY * 0.10

print(f"Soglie Alert simulate: Warning > {SOGLIA_WARNING:.0f}, Alarm > {SOGLIA_ALARM:.0f} crimini/giorno")

## 4. Interrogazioni Analitiche

Eseguiamo le analisi richieste utilizzando `query_api` e visualizzando i risultati con `matplotlib`.

In [ ]:
# Q1: Numero medio giornaliero di crimini per Area e Stagione [cite: 24]
# Logica: Contiamo i crimini totali per Area/Stagione e dividiamo per una stima dei giorni (5 anni * 90 giorni).
print("Esecuzione Q1: Media crimini giornalieri per Area e Stagione...")

q1 = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crimes")
  |> filter(fn: (r) => r["_tag"] == "DR_NO")
  |> group(columns: ["AREA_NAME", "Season"])
  |> count()
'''
result_q1 = query_api.query_data_frame(q1)

if not result_q1.empty:
    # Stima giorni per stagione in 5 anni ~ 450 giorni
    result_q1['daily_avg'] = result_q1['_value'] / 450 
    pivot_q1 = result_q1.pivot(index='AREA_NAME', columns='Season', values='daily_avg')
    
    pivot_q1.plot(kind='bar', figsize=(15, 6), width=0.8)
    plt.title('Q1: Media Giornaliera Crimini per Area e Stagione')
    plt.ylabel('Media Crimini / Giorno')
    plt.tight_layout()
    plt.show()
    
    # Analisi testuale [cite: 25]
    print("Analisi: Osservare il grafico per identificare le aree con picchi specifici in Estate o Inverno.")

In [ ]:
# Q2: Distribuzione Part 1 vs Part 2 per Day/Night (Mese: Luglio 2023) [cite: 26]
print("Esecuzione Q2: Distribuzione Day/Night (Luglio 2023)...")

q2 = f'''
from(bucket: "{bucket}")
  |> range(start: 2023-07-01T00:00:00Z, stop: 2023-07-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crimes")
  |> filter(fn: (r) => r["_field"] == "DR_NO")
  |> group(columns: ["Part 1-2", "DayNight"])
  |> count()
'''
result_q2 = query_api.query_data_frame(q2)

if not result_q2.empty:
    pivot_q2 = result_q2.pivot(index='DayNight', columns='Part 1-2', values='_value')
    pivot_q2.plot(kind='bar', figsize=(8, 5))
    plt.title('Q2: Crimini Part 1 vs Part 2 (Luglio 2023)')
    plt.ylabel('Totale Crimini')
    plt.show()
    print("Analisi: Verificare se i crimini gravi (Part 1) sono più frequenti di 'Night'[cite: 27].")

In [ ]:
# Q3: Giornata con maggior numero di crimini violenti per Area (CORRETTA)
print("\nEsecuzione Q3: Max crimini violenti giornalieri per Area...")

q3 = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crimes")
  |> filter(fn: (r) => r["Part 1-2"] == "1")
  |> filter(fn: (r) => r["_field"] == "DR_NO")
  |> group(columns: ["Area_Name"])
  |> aggregateWindow(every: 1d, fn: count)  // Calcola il conteggio giornaliero e mantiene il _time
  |> max()                                  // Trova il giorno con il valore massimo per ogni Area
'''

try:
    result_q3 = query_api.query_data_frame(q3)
    if not result_q3.empty:
        # Pulizia e visualizzazione
        # Rinominiamo le colonne per chiarezza
        display_q3 = result_q3[['Area_Name', '_time', '_value']].copy()
        display_q3['_time'] = pd.to_datetime(display_q3['_time']).dt.date
        display_q3.columns = ['Area', 'Data del Picco', 'Numero Crimini']
        
        # Ordiniamo per numero di crimini decrescente
        display_q3 = display_q3.sort_values('Numero Crimini', ascending=False)
        
        print(display_q3.head(10))
    else:
        print("Nessun dato trovato per Q3.")
except Exception as e:
    print(f"Errore Q3: {e}")

In [ ]:
# Q4: Età media vittime per Categoria e Anno [cite: 30]
print("Esecuzione Q4: Età media vittime...")

q4 = f'''
from(bucket: "{bucket}")
  |> range(start: 2020-01-01T00:00:00Z, stop: 2025-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "crimes")
  |> filter(fn: (r) => r["_field"] == "Vict_Age")
  |> filter(fn: (r) => r["_value"] > 0)
  |> group(columns: ["Part 1-2"])
  |> window(every: 1y)
  |> mean()
  |> duplicate(column: "_start", as: "Year")
  |> window(every: inf)
'''
result_q4 = query_api.query_data_frame(q4)

if not result_q4.empty:
    result_q4['Year'] = pd.to_datetime(result_q4['Year']).dt.year
    pivot_q4 = result_q4.pivot(index='Year', columns='Part 1-2', values='_value')
    print(pivot_q4)
    pivot_q4.plot(kind='line', marker='o', figsize=(10, 5))
    plt.title('Q4: Età Media Vittime per Anno')
    plt.ylabel('Età Media')
    plt.show()

### Performance Check [cite: 31, 32]
Confrontiamo il tempo di esecuzione di una query globale vs una filtrata per stagione.

In [ ]:
def measure_query(season_filter=None):
    base_q = f'''
    from(bucket: "{bucket}")
      |> range(start: 2023-01-01T00:00:00Z, stop: 2023-12-31T23:59:59Z)
      |> filter(fn: (r) => r["_measurement"] == "crimes")
      |> filter(fn: (r) => r["_field"] == "DR_NO")
    '''
    if season_filter:
        base_q += f' |> filter(fn: (r) => r["Season"] == "{season_filter}")'
    
    base_q += ' |> count()'
    
    start = time.time()
    query_api.query(base_q)
    return time.time() - start

t_global = measure_query()
t_seasonal = measure_query("Summer")

print(f"Tempo Query Globale: {t_global:.4f} s")
print(f"Tempo Query Stagionale (Summer): {t_seasonal:.4f} s")
print(f"Miglioramento: {((t_global - t_seasonal)/t_global)*100:.1f}% (Atteso: I tag indicizzati rendono il filtro veloce)")

--- 
## 5. Analisi Avanzata: Clustering Incrementale

Utilizziamo `MiniBatchKMeans` per raggruppare le aree in base ai pattern di criminalità aggregati (Area, Stagione, Anno).
Features: Totale Crimini, % Arma, % Notte, % Violenti[cite: 33].

In [ ]:
# Aggregazione dati (Dataset Pandas) per preparazione Features
df['Year'] = df['DATE OCC'].dt.year

agg_df = df.groupby(['Area_Name', 'Season', 'Year']).agg(
    Total_Crimes=('DR_NO', 'count'),
    Weapon_Pct=('WeaponFlag', 'mean'),
    Night_Pct=('DayNight', lambda x: (x=='Night').mean()),
    Violent_Pct=('Part 1-2', lambda x: (x=='1').mean())
).reset_index()

features = ['Total_Crimes', 'Weapon_Pct', 'Night_Pct', 'Violent_Pct']
X = agg_df[features]

# Normalizzazione
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Clustering Incrementale (Train su 2020-2024)
train_mask = agg_df['Year'] < 2025
X_train = X_scaled[train_mask]
X_test = X_scaled[~train_mask]

kmeans = MiniBatchKMeans(n_clusters=4, random_state=42, batch_size=256)
kmeans.fit(X_train)

# Assegnazione Cluster
agg_df.loc[train_mask, 'Cluster'] = kmeans.labels_
if len(X_test) > 0:
    agg_df.loc[~train_mask, 'Cluster'] = kmeans.predict(X_test)

# Analisi Risultati
print("--- Centroidi dei Cluster (Pattern identificati) ---")
centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print(centroids)

print("\n--- Confronto 2020 vs 2025: Evoluzione Cluster [cite: 34] ---")
cluster_dist = agg_df.groupby(['Year', 'Cluster']).size().unstack(fill_value=0)
print(cluster_dist)

# Visualizzazione
cluster_dist.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='viridis')
plt.title('Distribuzione Aree nei Cluster per Anno')
plt.ylabel('Numero di Aree')
plt.show()